In [11]:
import pandas as pd
import numpy as np

train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print(train.head(2))
print(test.head(2))

Train shape: (159571, 8)
Test shape: (153164, 2)
                 id                                       comment_text  toxic  \
0  0000997932d777bf  Explanation\nWhy the edits made under my usern...      0   
1  000103f0d9cfb60f  D'aww! He matches this background colour I'm s...      0   

   severe_toxic  obscene  threat  insult  identity_hate  
0             0        0       0       0              0  
1             0        0       0       0              0  
                 id                                       comment_text
0  00001cee341fdb12  Yo bitch Ja Rule is more succesful then you'll...
1  0000247867823ef7  == From RfC == \n\n The title is fine as it is...


In [12]:
drop_cols = ['id']

train.drop(columns=drop_cols, inplace=True)
test_ids = test['id']
test.drop(columns=drop_cols, inplace=True)

import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

train['comment_text'] = train['comment_text'].apply(clean_text)
test['comment_text'] = test['comment_text'].apply(clean_text)

In [19]:
from gensim.models import FastText
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import roc_auc_score
import os

def sentence_vector(text, model):
    words = [word for word in text.split() if word in model.wv]
    if not words:
        return np.zeros(model.vector_size)
    return np.max(model.wv[words], axis=0)

# def sentence_vector(text, model):
#     words = [word for word in text.split() if word in model.wv]
#     if not words:
#         return np.zeros(model.vector_size)
#     return np.mean(model.wv[words], axis=0)

X_train, X_val = train_test_split(train, test_size=0.2, random_state=42)
y_train = X_train.drop(columns=['comment_text'])
y_val = X_val.drop(columns=['comment_text'])
X_train = X_train['comment_text']
X_val = X_val['comment_text']

tokens = 0
sentences = []
for text in X_train:
    sentences.append(text.split())
    tokens += len(sentences[-1])

# print(f"Total sentences: {len(sentences)}, Total tokens: {tokens}")

# fasttext = FastText(
#     sentences=sentences,
#     vector_size=300, 
#     window=10, 
#     min_count=5, 
#     workers=os.cpu_count(), 
#     sg=1,
#     epochs=10,
#     seed=42,
# )
# fasttext.save('models/fasttext_model_v1.model')

fasttext = FastText.load('models/fasttext_model_v1.model')

X_train = np.array([sentence_vector(text, fasttext) for text in X_train])
X_val = np.array([sentence_vector(text, fasttext) for text in X_val])

print(X_train.shape)

model_lr = OneVsRestClassifier(LogisticRegression(
    max_iter=1000, 
    solver='liblinear',
    # n_jobs=-1, 
    random_state=42
))
model_lr.fit(X_train, y_train)
val_preds = model_lr.predict_proba(X_val)
auc_score = roc_auc_score(y_val, val_preds, average='macro')
print(f"Validation ROC AUC Score: {auc_score:.4f}")

(127656, 300)
Validation ROC AUC Score: 0.9727


In [ ]:
X_train = train
y_train = X_train.drop(columns=['comment_text'])
X_train = X_train['comment_text']
X_test = test['comment_text']

tokens = 0
sentences = []
for text in X_train:
    sentences.append(text.split())
    tokens += len(sentences[-1])

print(f"Total sentences: {len(sentences)}, Total tokens: {tokens}")

fasttext = FastText(
    sentences=sentences,
    vector_size=600, 
    window=10, 
    min_count=5, 
    workers=os.cpu_count(), 
    sg=1,
    epochs=20,
    seed=42,
)
fasttext.save('models/fasttext_model_v2.model')

fasttext = FastText.load('models/fasttext_model_v2.model')

X_train = np.array([sentence_vector(text, fasttext) for text in X_train])
X_test = np.array([sentence_vector(text, fasttext) for text in X_test])

model_lr = OneVsRestClassifier(LogisticRegression(
    max_iter=1000, 
    solver='liblinear',
    # n_jobs=-1, 
    random_state=42
))
model_lr.fit(X_train, y_train)
preds = model_lr.predict_proba(X_test)

Total sentences: 159571, Total tokens: 10553394


In [ ]:
submission = pd.DataFrame(preds, columns=y_train.columns)
submission.insert(0, "id", test_ids)
submission.to_csv('submissions/fasttext_logreg_submission_v2.csv', index=False)